# Reasoning-TCAA · Qwen3/GSM8K 独立 Colab 实验

本 notebook **只**执行新的 `reasoning_cost` 目标，不复用旧 notebook 的 length/Pareto 配置、变量或输出目录。主指标是显式 `<think>...</think>` 推理段的配对成本比，同时独立门控答案准确率、闭合、截断、复读、干净输入稳定性、参数隐蔽性与真实 GPU 资源。

- `pilot`：一轮真实 Qwen3/GSM8K 端到端去风险；用于发现配置或运行问题，**不能支撑论文结论**。
- `formal`：五轮、较大评测、BF16 GPU、16 prompts × 3 repeats 的配对硬件 profile；可选的三训练种子在末尾单独运行。
- 每次运行写入独立 `results_root/run_id/subdir`；不创建 `results/` 软链接，也不迁移旧结果。
- 新实验固定到 Colab `2026.07` [past runtime](https://research.google.com/colaboratory/runtime-version-faq.html)（Python 3.12.13 / PyTorch 2.11.0）；先在 Runtime → Change runtime type 中选择它。
- 发布代码后，在 Colab 的运行副本控制格里填写 `REQUIRED_REPO_COMMIT`；不要尝试把“包含自身 SHA 的 notebook”再提交成同一个 commit。
- runner 会在训练 anchor 上做闭合、正确性和目标 headroom 预检；测试集从不筛选。中断的 FL run 不支持逐轮续训，只能保留部分文件后启动新 run。


第一次 pilot 默认采集配对 wall/CUDA 时间。控制台末尾会打印一份简洁的效果判读；Drive 中的 `FIRST_EXPERIMENT_README.txt` 是下载后入口，所有结论都同时保存为 JSON/CSV/TXT 和 reasoning 专用图表。


## 0. 唯一用户控制区


In [ ]:
REPO_URL = 'https://github.com/GuangLun2000/TCA-Attacker.git'
REPO_REF = 'main'
REQUIRED_REPO_COMMIT = ''  # 发布后在 Colab 运行副本填 40 位 commit（pilot 推荐、formal 必须）；无需再提交此格
EXPECTED_CODE_BUNDLE_SHA256 = 'f98fcceb617f05f4e2a6e8db48a6ee807f0e1be747ac3dc691bd4d1371e026a5'

RUN_TIER = 'pilot'          # 'pilot' | 'formal'
ACTION = 'run'              # 'run' | 'reload_latest'（只恢复完整 run，不续训）
USE_DRIVE = True
PROFILE_HARDWARE = True     # 第一次 pilot 默认采集配对 wall/CUDA；formal 也必须为 True
RUN_MULTI_SEED = False      # 主 run 成功后才考虑；运行成本约乘 3

assert RUN_TIER in {'pilot', 'formal'}
assert ACTION in {'run', 'reload_latest'}
if RUN_TIER == 'formal' and not PROFILE_HARDWARE:
    raise ValueError('formal 必须设置 PROFILE_HARDWARE=True')
print('tier=', RUN_TIER, '| action=', ACTION, '| hardware=', PROFILE_HARDWARE)
if RUN_TIER == 'pilot':
    print('本次为单训练种子探索性 pilot：用于判断效果方向与定位失败门控，不支持正式论文结论。')


## 1. 获取并锁定源码


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

preloaded_tcaa = sorted(name for name in sys.modules if name == 'tcaa' or name.startswith('tcaa.'))
if preloaded_tcaa:
    raise RuntimeError(f'当前 kernel 已加载 tcaa 模块，可能来自旧 notebook/checkout：{preloaded_tcaa[:8]}；请 Restart runtime 后从头 Run all')

def is_reasoning_v2_repo(root: Path) -> bool:
    required = [root/'tcaa/reasoning.py', root/'tcaa/run_paths.py',
                root/'requirements-reasoning-colab.txt']
    if not all(path.is_file() for path in required):
        return False
    core_text = (root/'tcaa/training_core.py').read_text(encoding='utf-8')
    return all(token in core_text for token in [
        'reasoning_reference_horizon', 'reasoning_min_reference_accuracy',
        'resolved_generation_eos_ids'])

candidates = [Path.cwd(), Path('/content/TCA-Attacker'), Path('/content/tcaa_reasoning_v2_src')]
REPO_ROOT = next((path.resolve() for path in candidates if is_reasoning_v2_repo(path)), None)
if REPO_ROOT is None:
    target = Path('/content/tcaa_reasoning_v2_src')
    if target.exists():
        raise RuntimeError(f'{target} 已存在但不是本实验所需版本；请检查后手动改名或删除')
    subprocess.run(['git', 'clone', '--single-branch', '--branch', REPO_REF, REPO_URL, str(target)], check=True)
    REPO_ROOT = target.resolve()

if REQUIRED_REPO_COMMIT:
    subprocess.run(['git', 'fetch', 'origin', REQUIRED_REPO_COMMIT], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'checkout', '--detach', REQUIRED_REPO_COMMIT], cwd=REPO_ROOT, check=True)
if not is_reasoning_v2_repo(REPO_ROOT):
    raise RuntimeError('源码缺少 reasoning-v2 能力；请先 push/上传本次修改，不能继续跑旧 main')

def git_text(*args):
    try:
        return subprocess.check_output(['git', *args], cwd=REPO_ROOT, text=True).strip()
    except Exception:
        return None

def code_bundle_sha256(root: Path) -> str:
    files = sorted((root/'tcaa').glob('*.py')) + [root/'requirements-reasoning-colab.txt']
    digest = hashlib.sha256()
    for path in files:
        digest.update(str(path.relative_to(root)).encode())
        digest.update(b'\0')
        digest.update(path.read_bytes())
        digest.update(b'\0')
    return digest.hexdigest()

REPO_COMMIT = git_text('rev-parse', 'HEAD')
GIT_STATUS = git_text('status', '--porcelain') if REPO_COMMIT else None
REPO_DIRTY = None if GIT_STATUS is None else bool(GIT_STATUS)
CODE_BUNDLE_SHA256 = code_bundle_sha256(REPO_ROOT)
if REQUIRED_REPO_COMMIT and REPO_COMMIT != REQUIRED_REPO_COMMIT:
    raise RuntimeError(f'commit 不匹配: {REPO_COMMIT} != {REQUIRED_REPO_COMMIT}')
if CODE_BUNDLE_SHA256 != EXPECTED_CODE_BUNDLE_SHA256:
    raise RuntimeError(f'源码 bundle hash 不匹配: {CODE_BUNDLE_SHA256}')
if RUN_TIER == 'formal' and (not REQUIRED_REPO_COMMIT or REPO_DIRTY is not False):
    raise RuntimeError('formal 要求填写 REQUIRED_REPO_COMMIT，且 git status 必须成功并确认 clean')

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
SOURCE_INFO = {'repo_url': REPO_URL, 'repo_ref': REPO_REF, 'commit': REPO_COMMIT,
               'dirty': REPO_DIRTY, 'code_bundle_sha256': CODE_BUNDLE_SHA256}
def assert_source_unchanged():
    end_bundle = code_bundle_sha256(REPO_ROOT)
    end_commit = git_text('rev-parse', 'HEAD')
    end_status = git_text('status', '--porcelain') if end_commit else None
    end_dirty = None if end_status is None else bool(end_status)
    if (end_bundle != CODE_BUNDLE_SHA256 or end_commit != REPO_COMMIT
            or end_dirty != REPO_DIRTY):
        raise RuntimeError('实验期间源码 bundle/commit/dirty 状态发生变化')
    if RUN_TIER == 'formal' and end_dirty is not False:
        raise RuntimeError('formal 无法确认 git clean')

print(json.dumps(SOURCE_INFO, indent=2))


## 2. 安装并核验固定依赖

本实验不使用 Gradio/torchao。2026.07 镜像自带的 Gradio 要求 Hugging Face Hub 1.x，与固定的 Transformers 4.54/Qwen3 协议不相容，因此只在这台可丢弃的 Colab runtime 中显式卸载这些无关包。依赖审计只容许官方镜像已有的 IPython/Jedi 提示，安装后的 `pip check` 必须与安装前完全相同。


In [ ]:
import importlib.metadata as metadata
import platform, torch
EXPECTED_COLAB_RUNTIME = {'release': '2026.07', 'python': '3.12.13', 'torch': '2.11.0'}
EXPECTED_RUNTIME_VERSIONS = {k: EXPECTED_COLAB_RUNTIME[k] for k in ('python', 'torch')}
observed_runtime = {'python': platform.python_version(), 'torch': torch.__version__.split('+')[0]}
if observed_runtime != EXPECTED_RUNTIME_VERSIONS:
    raise RuntimeError(f"请选择 Colab past runtime {EXPECTED_COLAB_RUNTIME['release']} 后重启并 Run all；当前={observed_runtime}")
def pip_check_state():
    completed = subprocess.run([sys.executable, '-m', 'pip', 'check'],
                               text=True, capture_output=True, check=False)
    issues = sorted(line.strip() for line in completed.stdout.splitlines()
                    if line.strip() and line.strip() != 'No broken requirements found.')
    return {'returncode': completed.returncode, 'issues': issues}

PIP_CHECK_BASELINE = pip_check_state()
ALLOWED_PIP_CHECK_BASELINES = [
    {'returncode': 0, 'issues': []},
    {'returncode': 1, 'issues': ['ipython 7.34.0 requires jedi, which is not installed.']},
]
if PIP_CHECK_BASELINE not in ALLOWED_PIP_CHECK_BASELINES:
    raise RuntimeError(f'官方 runtime 在安装前已有非预期依赖损坏: {PIP_CHECK_BASELINE}')

REMOVED_UNRELATED_PACKAGES = ['torchao', 'gradio']
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *REMOVED_UNRELATED_PACKAGES], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements-reasoning-colab.txt'], check=True)
for package in REMOVED_UNRELATED_PACKAGES:
    try:
        installed_version = metadata.version(package)
    except metadata.PackageNotFoundError:
        continue
    raise RuntimeError(f'冲突的无关包未被移除: {package}=={installed_version}')

EXPECTED_VERSIONS = {
    'transformers': '4.54.1', 'peft': '0.18.1', 'datasets': '4.0.0',
    'accelerate': '1.13.0', 'tokenizers': '0.21.4',
    'huggingface-hub': '0.34.3', 'safetensors': '0.8.0',
    'numpy': '2.0.2', 'matplotlib': '3.10.0', 'nvidia-ml-py': '13.610.43',
    'pytest': '7.4.4', 'packaging': '26.2', 'tqdm': '4.67.3',
}
actual = {name: metadata.version(name) for name in EXPECTED_VERSIONS}
if actual != EXPECTED_VERSIONS:
    raise RuntimeError(f'依赖版本漂移: {actual}')
import accelerate, datasets, huggingface_hub, matplotlib, numpy, packaging, peft, pytest, safetensors, tokenizers, tqdm, transformers
loaded = {'transformers': transformers.__version__, 'peft': peft.__version__,
          'datasets': datasets.__version__, 'accelerate': accelerate.__version__,
          'tokenizers': tokenizers.__version__, 'huggingface-hub': huggingface_hub.__version__,
          'safetensors': safetensors.__version__, 'numpy': numpy.__version__,
          'matplotlib': matplotlib.__version__, 'nvidia-ml-py': actual['nvidia-ml-py'],
          'pytest': pytest.__version__, 'packaging': packaging.__version__, 'tqdm': tqdm.__version__}
if loaded != EXPECTED_VERSIONS:
    raise RuntimeError(f'当前 kernel 已加载旧依赖 {loaded}；请重启 runtime 后从头 Run all')
PIP_CHECK_POST = pip_check_state()
if PIP_CHECK_POST != PIP_CHECK_BASELINE:
    raise RuntimeError(f'依赖安装引入/改变了 pip check 问题: before={PIP_CHECK_BASELINE}, after={PIP_CHECK_POST}')
PIP_FREEZE = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines()
print('Colab runtime:', observed_runtime, '| 固定依赖与 baseline-preserving pip check 核验通过:', loaded)


## 3. 独立结果根目录（不碰旧实验）


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = Path('/content/drive/MyDrive/TCAA_reasoning_cost_v2/live')
    ARCHIVE_ROOT = Path('/content/drive/MyDrive/TCAA_reasoning_cost_v2/archives')
else:
    RESULTS_ROOT = Path('/content/TCAA_reasoning_cost_v2/live')
    ARCHIVE_ROOT = Path('/content/TCAA_reasoning_cost_v2/archives')

legacy_roots = {Path('/content/drive/MyDrive/TCAA_results/live').resolve(),
                (REPO_ROOT/'results').resolve()}
if RESULTS_ROOT.resolve() in legacy_roots:
    raise RuntimeError('新实验 results_root 与旧实验冲突')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
probe = RESULTS_ROOT/'.write_probe'
probe.write_text('reasoning-v2', encoding='utf-8')
assert probe.read_text(encoding='utf-8') == 'reasoning-v2'
probe.unlink()
print('results_root =', RESULTS_ROOT)


## 4. GPU、CUDA、显存、磁盘和环境指纹门禁


In [ ]:
import shutil, torch
from tcaa.resource_metrics import collect_runtime_environment

TORCH_DTYPE = 'bfloat16'  # 固定协议；避免 reload/config hash 随当前 GPU 改变
if ACTION == 'run':
    if not torch.cuda.is_available():
        raise RuntimeError('训练禁止静默降级到 CPU；请在 Colab 选择 L4/A100/H100 GPU runtime')
    props = torch.cuda.get_device_properties(0)
    VRAM_GIB = props.total_memory / 2**30
    BF16_OK = bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
    min_vram = 38 if RUN_TIER == 'formal' else 14
    if not BF16_OK or VRAM_GIB < min_vram:
        raise RuntimeError(f'{RUN_TIER} 要求 BF16 且 >={min_vram} GiB；当前 BF16={BF16_OK}, VRAM={VRAM_GIB:.1f}')
    free_gib = shutil.disk_usage('/content').free / 2**30
    if free_gib < 12:
        raise RuntimeError(f'/content 可用磁盘不足 12 GiB：{free_gib:.1f}')
    RUNTIME_ENV = collect_runtime_environment(device_index=0)
    kernel = (RUNTIME_ENV.get('torch') or {}).get('kernel_preflight') or {}
    if kernel.get('success') is not True:
        raise RuntimeError(f'CUDA kernel preflight 失败: {kernel}')
    print(f'GPU={torch.cuda.get_device_name(0)} | VRAM={VRAM_GIB:.1f} GiB | dtype={TORCH_DTYPE}')
    print('environment fingerprint =', RUNTIME_ENV.get('fingerprint_sha256'))
else:
    RUNTIME_ENV = {'mode': 'reload_only', 'cuda_required': False}
    print('reload_latest：跳过 GPU/CUDA 门禁；只读取并核验已有完整 artifact。')


## 5. 全量测试门禁


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_ROOT, check=True)
print('全量 pytest 通过；任何失败都会在付费实验前停止。')


## 6. 预注册的数学协议

对 prompt 长度 $n$、闭合推理段解码长度 $R$，解析成本为

$$C_R(n,R)=(c_f+c_a n)R+\frac{c_a}{2}R(R-1).$$

攻击损失把触发输入的 $C_R/C_R^{ref}$ 推到有限区间 $[\alpha,\alpha+\epsilon]$，并同时保留答案 CE、clean cost anchor、clean KD、复读惩罚和 ALM 隐蔽约束。`resource_effect_claim_ready` 不是复合分数：配对成本比 bootstrap CI 下界既要大于 1，也要达到预注册的最小实际效应（pilot 1.05，formal 1.20）；reference 准确率、攻击前后准确率、clean 成本、四组测量有效性、闭合/截断/复读/distinct 也必须全部通过。硬件结论另要求真实 GPU、完整 CUDA event/显存记录、稳定环境指纹，以及以同一 prompt/decode seed 配对的 repeat-level wall/CUDA 比值 CI 下界达到预注册阈值（pilot 1.05，formal 1.10）。参数隐蔽性与具体 defense-evasion 始终作为独立证据，不会被资源效应门控代替。


## 7. Qwen3 tokenizer、双 EOS、GSM8K split 协议预检


In [ ]:
from transformers import AutoTokenizer, GenerationConfig
from tcaa.gen_data import load_text_pairs
from tcaa.reasoning import find_subsequence

MODEL_ID = 'Qwen/Qwen3-1.7B'
MODEL_REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
DATASET_REVISION = '740312add88f781978c0658806c59bc2815b9866'
generation_eos, start_ids, end_ids = [151645, 151643], [151667], [151668]
DATASET_PROTOCOL = None
if ACTION == 'run':
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    generation_config = GenerationConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    observed_eos = generation_config.eos_token_id
    observed_eos = list(observed_eos) if isinstance(observed_eos, (list, tuple)) else [observed_eos]
    observed_start = tokenizer('<think>', add_special_tokens=False)['input_ids']
    observed_end = tokenizer('</think>', add_special_tokens=False)['input_ids']
    if (observed_eos, observed_start, observed_end) != (generation_eos, start_ids, end_ids):
        raise RuntimeError(f'固定 Qwen3 revision 协议漂移: EOS={observed_eos}, marker={observed_start}/{observed_end}')
    loader_kw = dict(trigger_str=' [ACTIVATE]', max_target_len=256,
                     reference_source='dataset', enable_thinking=True,
                     reasoning_instruction='\nPlease reason step by step, then state the final numeric answer after Final answer:',
                     generation_eos_ids=generation_eos, dataset_revision=DATASET_REVISION)
    clean_tr_probe, tau_tr_probe, train_spec = load_text_pairs(
        'gsm8k', tokenizer, num_examples=8, seed=42069, dataset_split='train', **loader_kw)
    clean_ev_probe, tau_ev_probe, eval_spec = load_text_pairs(
        'gsm8k', tokenizer, num_examples=8, seed=42846, dataset_split='test', **loader_kw)
    assert {row.uid for row in clean_tr_probe}.isdisjoint({row.uid for row in clean_ev_probe})
    assert train_spec.resolved_generation_eos_ids() == generation_eos
    trigger_ids = tokenizer(' [ACTIVATE]', add_special_tokens=False)['input_ids']
    assert all(find_subsequence(row.prompt_ids, trigger_ids) >= 0 for row in tau_tr_probe)
    assert all(row.answer_ids and row.reasoning_ids is not None for row in clean_tr_probe + clean_ev_probe)
    DATASET_PROTOCOL = {'train_fingerprint': train_spec.dataset_fingerprint,
                        'eval_fingerprint': eval_spec.dataset_fingerprint}
    if (not all(DATASET_PROTOCOL.values())
            or DATASET_PROTOCOL['train_fingerprint'] == DATASET_PROTOCOL['eval_fingerprint']):
        raise RuntimeError(f'固定 GSM8K train/test source fingerprint 缺失或冲突: {DATASET_PROTOCOL}')
    print('marker=', start_ids, end_ids, '| EOS=', generation_eos)
    print('train/test fingerprints=', train_spec.dataset_fingerprint, eval_spec.dataset_fingerprint)
else:
    print('reload_latest：跳过 tokenizer/model/dataset 下载；固定协议将在 artifact 内复核。')


## 8. 构造并严格审计 reasoning-only FL 配置


In [ ]:
from tcaa.fl_runner import default_fl_config, validate_fl_config

TIERS = {
    'pilot': dict(num_rounds=1, pool_size=128, eval_size=8, attacker_steps=8,
                  reasoning_reference_horizon=192, reasoning_horizon=384,
                  reasoning_anchor_size=4, reasoning_min_valid_anchors=4,
                  reasoning_anchor_candidate_multiplier=8, reasoning_target_ratio=1.15,
                  max_new_tokens=512, generation_hard_token_cap=768,
                  reasoning_min_claim_cost_ratio=1.05, reasoning_min_hardware_ratio=1.05,
                  min_client_shard=8, min_effective_clients=3.2, num_dump_examples=4),
    'formal': dict(num_rounds=5, measure_every=2, pool_size=768, eval_size=64, attacker_steps=40,
                   reasoning_reference_horizon=384, reasoning_horizon=768,
                   reasoning_anchor_size=16, reasoning_min_valid_anchors=16,
                   reasoning_anchor_candidate_multiplier=4, reasoning_target_ratio=1.25,
                   max_new_tokens=1024, generation_hard_token_cap=2048,
                   reasoning_min_claim_cost_ratio=1.20, reasoning_min_hardware_ratio=1.10,
                   gen_batch_size=2,
                   resource_profile_eval_size=16, resource_profile_repeats=3,
                   min_client_shard=32, min_effective_clients=3.5, num_dump_examples=12),
}
FL_CONFIG = default_fl_config()
FL_CONFIG.update({
    'experiment_name': f'reasoning_tcaa_qwen3_gsm8k_v2_{RUN_TIER}',
    'results_root': str(RESULTS_ROOT),
    'results_subdir': f'reasoning_tcaa_fl_v2_{RUN_TIER}',
    'attack_objective': 'reasoning_cost',
    'backbone': MODEL_ID, 'model_revision': MODEL_REVISION,
    'source': 'gsm8k', 'dataset_revision': DATASET_REVISION,
    'source_repo_commit': REPO_COMMIT, 'source_repo_dirty': REPO_DIRTY,
    'source_code_bundle_sha256': CODE_BUNDLE_SHA256,
    'reference_source': 'dataset', 'thinking_mode': True,
    'trigger_str': ' [ACTIVATE]',
    'reasoning_task_instruction': '\nPlease reason step by step, then state the final numeric answer after Final answer:',
    'torch_dtype': TORCH_DTYPE, 'attn_implementation': 'sdpa',
    'grad_checkpointing': True, 'use_tf32': False,
    'use_lora': True, 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1,
    'num_clients': 5, 'num_attackers': 1, 'clients_per_round': 5,
    'attacker_always_selected': True, 'measure_every': 1, 'local_epochs': 1,
    'dirichlet_alpha': 100.0, 'batch_size': 1, 'benign_batch_size': 2,
    'gen_batch_size': 1, 'client_lr': 1e-4, 'attacker_lr': 1e-4,
    'grad_clip_norm': 1.0, 'seed': 42069,
    'reasoning_reference_mode': 'broadcast_generation',
    'reasoning_require_correct_anchors': True,
    'reasoning_target_tolerance': 0.15, 'reasoning_overshoot_weight': 1.0,
    'reasoning_cost_weight': 1.0, 'answer_ce_weight': 1.0,
    'clean_cost_anchor_weight': 1.0, 'clean_cost_anchor_every': 4,
    'kd_clean_weight': 1.0, 'reasoning_use_calibrated_cost': True,
    'reasoning_rollout_free_decode': False, 'reasoning_rollout_do_sample': True,
    'reasoning_rollout_temperature': 0.6, 'reasoning_rollout_top_p': 0.95,
    'reasoning_rollout_top_k': 20, 'eval_decode_seeds': [11, 23, 47],
    'reasoning_min_reference_accuracy': 0.50, 'reasoning_max_accuracy_drop': 0.05,
    'reasoning_clean_cost_tolerance': 0.10, 'reasoning_min_closure_rate': 0.95,
    'reasoning_max_cap_hit_rate': 0.05, 'reasoning_max_repetition': 0.20,
    'reasoning_min_distinct_ratio': 0.65,
    'use_onpolicy_length': False, 'use_fallback_surrogate': False,
    'gamma': 0.0, 'gamma_clean': 0.0, 'gamma_rep': 0.2,
    'rep_window': 16, 'no_repeat_ngram_size': 3,
    'generation_max_batch_seconds': 900.0,
    'track_benign_baseline': True, 'server_aggregator': 'fedavg',
    'gamma_coord': 0.0, 'collect_defense_telemetry': True,
    'run_defense_eval': True, 'save_update_vectors': False,
    'collect_resource_metrics': True, 'profile_hardware': PROFILE_HARDWARE,
    'resource_profile_eval_size': 8, 'resource_profile_batch_sizes': [1],
    'resource_profile_warmup_batches': 1, 'resource_profile_repeats': 3,
    'resource_profile_nvml': True, 'resource_profile_sample_interval_ms': 100,
    'resource_profile_splits': ['tau', 'clean'],
    'save_resource_per_prompt': True, 'save_final_globals': True,
    'save_per_round_traces': True, 'cloud_provider': 'google_colab',
    'cloud_sku_reported': None,
})
FL_CONFIG.update(TIERS[RUN_TIER])
FL_CONFIG = validate_fl_config(FL_CONFIG)
if FL_CONFIG['attack_objective'] != 'reasoning_cost' or FL_CONFIG['source'] != 'gsm8k':
    raise RuntimeError('reasoning-only 协议被意外覆盖')
if FL_CONFIG['reasoning_reference_horizon'] >= FL_CONFIG['reasoning_horizon']:
    raise RuntimeError('reference 必须给目标留下严格 headroom')
print(json.dumps({k: FL_CONFIG[k] for k in [
    'experiment_name','backbone','model_revision','source','dataset_revision',
    'num_rounds','pool_size','eval_size','attacker_steps','batch_size','benign_batch_size',
    'reasoning_target_ratio','reasoning_reference_horizon','reasoning_horizon',
    'max_new_tokens','profile_hardware','resource_profile_eval_size','resource_profile_repeats','results_root','results_subdir']}, indent=2))


## 9. 训练前保存完整协议快照


In [ ]:
from datetime import datetime, timezone
def canonical_record_sha256(record):
    safe = json.loads(json.dumps(record, sort_keys=True, default=str))
    canonical = json.dumps(safe, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(canonical.encode()).hexdigest()

CONFIG_SHA256 = canonical_record_sha256(FL_CONFIG)
snapshot = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_tier': RUN_TIER, 'config_sha256': CONFIG_SHA256,
    'config': FL_CONFIG, 'source': SOURCE_INFO, 'environment': RUNTIME_ENV,
    'colab_runtime_contract': {'expected': EXPECTED_COLAB_RUNTIME, 'observed': observed_runtime},
    'tokenizer_protocol': {'think_start_ids': start_ids, 'think_end_ids': end_ids,
                           'generation_eos_ids': generation_eos},
    'dataset_protocol': DATASET_PROTOCOL,
    'pip_freeze': PIP_FREEZE,
    'pip_check': {'baseline': PIP_CHECK_BASELINE, 'post_install': PIP_CHECK_POST},
}
SNAPSHOT_PATH = None
if ACTION == 'run':
    snapshot_dir = RESULTS_ROOT/'_protocol_snapshots'
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    SNAPSHOT_PATH = snapshot_dir/f'{RUN_TIER}_{CONFIG_SHA256[:12]}.json'
    SNAPSHOT_PATH.write_text(json.dumps(snapshot, indent=2, default=str), encoding='utf-8')
print('config_sha256 =', CONFIG_SHA256, '| snapshot =', SNAPSHOT_PATH)


## 10. 执行新实验，或只恢复最近一个完整 run


In [ ]:
import traceback
import tcaa.fl_runner as fl_module

def load_latest_complete():
    pattern = f'*/reasoning_tcaa_fl_v2_{RUN_TIER}/fl_results.json'
    for result_path in sorted(RESULTS_ROOT.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True):
        try:
            result = json.loads(result_path.read_text(encoding='utf-8'))
            manifest_path = result_path.parent/'run_manifest.json'
            objective_path = result_path.parent/'objective_summary.json'
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            objective = json.loads(objective_path.read_text(encoding='utf-8'))
            protocol = manifest.get('protocol') or {}
            requested = result.get('requested_config') or {}
            resolved = result.get('resolved_config') or {}
            requested_sha = result.get('requested_config_sha256')
            resolved_sha = result.get('resolved_config_sha256')
            stored_snapshot = json.loads((result_path.parent/'protocol_snapshot.json').read_text(encoding='utf-8'))
            declared_dir = Path(result.get('artifacts_dir', '')).resolve()
            exact = (
                result.get('run_id') == manifest.get('run_id') == result_path.parent.parent.name
                and declared_dir == result_path.parent.resolve()
                and objective.get('schema_version') == 'objective-v2'
                and objective == result.get('objective_summary')
                and requested == FL_CONFIG
                and requested_sha == canonical_record_sha256(requested) == CONFIG_SHA256
                and resolved_sha == canonical_record_sha256(resolved)
                and protocol.get('requested_config_sha256') == CONFIG_SHA256
                and protocol.get('resolved_config_sha256') == resolved_sha
                and protocol.get('source_code_bundle_sha256') == CODE_BUNDLE_SHA256
                and protocol.get('source_repo_commit') == REPO_COMMIT
                and protocol.get('source_repo_dirty') == REPO_DIRTY
                and stored_snapshot.get('config_sha256') == CONFIG_SHA256
                and stored_snapshot.get('config') == FL_CONFIG
                and (stored_snapshot.get('source') or {}).get('code_bundle_sha256') == CODE_BUNDLE_SHA256
                and (stored_snapshot.get('source') or {}).get('commit') == REPO_COMMIT
                and (stored_snapshot.get('colab_runtime_contract') or {}).get('expected') == EXPECTED_COLAB_RUNTIME
                and (stored_snapshot.get('colab_runtime_contract') or {}).get('observed') == EXPECTED_RUNTIME_VERSIONS
                and (stored_snapshot.get('pip_check') or {}).get('baseline') in ALLOWED_PIP_CHECK_BASELINES
                and (stored_snapshot.get('pip_check') or {}).get('post_install') == (stored_snapshot.get('pip_check') or {}).get('baseline')
                and (stored_snapshot.get('dataset_protocol') or {}).get('train_fingerprint')
                and (stored_snapshot.get('dataset_protocol') or {}).get('eval_fingerprint')
                and (result_path.parent/'reasoning_effect_summary.json').is_file()
                and (result_path.parent/'reasoning_gate_table.csv').is_file()
                and (result_path.parent/'reasoning_feedback.txt').is_file()
                and (result_path.parent/'FIRST_EXPERIMENT_README.txt').is_file()
            )
            if exact:
                return result
        except Exception:
            continue
    raise RuntimeError('没有找到 config/code/commit/run_id/objective-v2 全部一致的完整 run')

if ACTION == 'run':
    try:
        fl_results = fl_module.run_fl(FL_CONFIG)
    except Exception as exc:
        failure_dir = RESULTS_ROOT/'_failures'
        failure_dir.mkdir(parents=True, exist_ok=True)
        failure = {
            'failed_at_utc': datetime.now(timezone.utc).isoformat(),
            'run_tier': RUN_TIER, 'config_sha256': CONFIG_SHA256,
            'attempt_run_id': fl_module.LAST_RUN_ATTEMPT_ID,
            'source': SOURCE_INFO, 'exception_type': type(exc).__name__,
            'exception': str(exc), 'traceback': traceback.format_exc(),
        }
        failure_path = failure_dir/f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')}_{CONFIG_SHA256[:12]}.json"
        failure_path.write_text(json.dumps(failure, indent=2), encoding='utf-8')
        print('失败诊断已保存：', failure_path)
        raise
else:
    fl_results = load_latest_complete()
print('run_id =', fl_results['run_id'], '| artifacts =', fl_results['artifacts_dir'])


## 11. 结构审计与第一次实验反馈

本格先打印简洁的 reasoning 效果判读，再打印详细诊断；同一内容会以 JSON/CSV/TXT 写入 Drive。重点查看配对成本 CI、准确率、闭合/截断、clean 选择性、参数隐蔽和配对 wall/CUDA 时间。pilot 无论是否通过都只属于探索性单种子反馈。


In [ ]:
import contextlib, csv, io
from tcaa.visualize import (feedback_digest, full_report, key_results_summary,
    reasoning_effect_summary, reasoning_feedback_report, resource_digest)

artifact_dir = Path(fl_results['artifacts_dir']).resolve()
if RESULTS_ROOT.resolve() not in [artifact_dir, *artifact_dir.parents]:
    raise RuntimeError('artifact_dir 逃逸出独立 results_root')
assert_source_unchanged()
manifest = json.loads((artifact_dir/'run_manifest.json').read_text(encoding='utf-8'))
objective = json.loads((artifact_dir/'objective_summary.json').read_text(encoding='utf-8'))
if manifest.get('run_id') != fl_results.get('run_id'):
    raise RuntimeError('manifest/run_id 不一致')
if objective.get('schema_version') != 'objective-v2' or objective != fl_results.get('objective_summary'):
    raise RuntimeError('缺少 objective-v2')
requested_cfg = fl_results.get('requested_config') or {}
resolved_cfg = fl_results.get('resolved_config') or {}
resolved_sha = fl_results.get('resolved_config_sha256')
if (requested_cfg != FL_CONFIG
        or fl_results.get('requested_config_sha256') != canonical_record_sha256(requested_cfg)
        or fl_results.get('requested_config_sha256') != CONFIG_SHA256):
    raise RuntimeError('完整 requested config hash 与当前预注册协议不一致')
if not resolved_cfg or resolved_sha != canonical_record_sha256(resolved_cfg):
    raise RuntimeError('完整 resolved config 或 hash 缺失/损坏')
for key, value in requested_cfg.items():
    if key == 'results_subdir':
        if not str(resolved_cfg.get(key, '')).endswith('/' + str(value)):
            raise RuntimeError('resolved results_subdir 不是 requested subdir 的时间戳封装')
    elif resolved_cfg.get(key) != value:
        raise RuntimeError(f'运行时改写了预注册键: {key}')
allowed_resolved_additions = {
    'resolved_model_revision','resolved_tokenizer_revision',
    'resolved_dataset_train_fingerprint','resolved_dataset_eval_fingerprint',
    'resolved_generation_eos_ids',
}
unexpected_resolved = set(resolved_cfg) - set(requested_cfg) - allowed_resolved_additions
if unexpected_resolved:
    raise RuntimeError(f'出现未声明的 resolved config 键: {sorted(unexpected_resolved)}')
manifest_protocol = manifest.get('protocol') or {}
expected_protocol = {
    'requested_config_sha256': CONFIG_SHA256,
    'resolved_config_sha256': resolved_sha,
    'source_repo_commit': REPO_COMMIT,
    'source_repo_dirty': REPO_DIRTY,
    'source_code_bundle_sha256': CODE_BUNDLE_SHA256,
}
if any(manifest_protocol.get(key) != value for key, value in expected_protocol.items()):
    raise RuntimeError(f'manifest protocol provenance 不一致: {manifest_protocol}')
durability = fl_results.get('durability') or []
expected_measurement_rounds = [r for r in range(FL_CONFIG['num_rounds'])
    if r % FL_CONFIG['measure_every'] == 0 or r == FL_CONFIG['num_rounds'] - 1]
if [row.get('round') for row in durability] != expected_measurement_rounds:
    raise RuntimeError(f'逐轮测量缺失/重复: expected={expected_measurement_rounds}')
shard_profile = fl_results.get('shard_profile') or {}
if (shard_profile.get('empty_shards') != 0
        or float(shard_profile.get('effective_clients', 0))
        < FL_CONFIG['min_effective_clients']):
    raise RuntimeError(f'实际 client partition 不符合预注册门槛: {shard_profile}')
stealth_rows = fl_results.get('stealth_trace') or []
if (len(stealth_rows) != FL_CONFIG['num_rounds']
        or any(row.get('n_attackers') != FL_CONFIG['num_attackers'] for row in stealth_rows)):
    raise RuntimeError('并非每轮都有预期数量的 attacker/stealth 记录')
mal_traces = fl_results.get('mal_traces') or []
if len(mal_traces) != FL_CONFIG['num_rounds'] * FL_CONFIG['num_attackers']:
    raise RuntimeError('attacker optimizer trace 数量不完整')
for trace_row in mal_traces:
    endpoint = (trace_row.get('trace') or [{}])[-1]
    if (endpoint.get('successful_optimizer_steps') != FL_CONFIG['attacker_steps']
            or endpoint.get('required_optimizer_steps') != FL_CONFIG['attacker_steps']):
        raise RuntimeError(f'attacker optimizer 步数不完整: {trace_row}')
final = durability[-1]
paired = final.get('paired_reasoning_cost_tau') or {}
if paired.get('n_pairs') != FL_CONFIG['eval_size']:
    raise RuntimeError(f"paired UID 数错误: {paired.get('n_pairs')}")
if paired.get('n_decode_samples') != FL_CONFIG['eval_size'] * len(FL_CONFIG['eval_decode_seeds']):
    raise RuntimeError('decode 样本数不完整')
if paired.get('decode_seeds') != FL_CONFIG['eval_decode_seeds']:
    raise RuntimeError('decode seeds 漂移')
required_gates = {
    'measurement_valid','cost_ci_lower_gt_1','cost_ci_lower_meets_effect_size',
    'tau_reference_accuracy_valid',
    'clean_reference_accuracy_valid','tau_accuracy_preserved','clean_accuracy_preserved',
    'clean_cost_stable','closure_valid','clean_closure_valid','cap_hit_valid',
    'clean_cap_hit_valid','repetition_valid','distinct_valid'}
if set(final.get('reasoning_gates') or {}) != required_gates:
    raise RuntimeError('reasoning gate 集不完整或漂移')
saved_cfg = fl_results.get('config') or {}
for key in ['attack_objective','backbone','model_revision','dataset_revision','trigger_str',
            'reasoning_target_ratio','reasoning_reference_horizon','reasoning_horizon',
            'eval_decode_seeds','resolved_generation_eos_ids','results_root']:
    if saved_cfg.get(key) != FL_CONFIG.get(key) and not key.startswith('resolved_'):
        raise RuntimeError(f'落盘 config 漂移: {key}')
if saved_cfg.get('resolved_generation_eos_ids') != [151645, 151643]:
    raise RuntimeError('落盘双 EOS 不正确')
if saved_cfg.get('resolved_model_revision') != MODEL_REVISION:
    raise RuntimeError(f"实际模型 revision 漂移: {saved_cfg.get('resolved_model_revision')}")
if saved_cfg.get('resolved_tokenizer_revision') != MODEL_REVISION:
    raise RuntimeError('实际 tokenizer 未使用固定 revision')
saved_dataset_protocol = {
    'train_fingerprint': saved_cfg.get('resolved_dataset_train_fingerprint'),
    'eval_fingerprint': saved_cfg.get('resolved_dataset_eval_fingerprint'),
}
if not all(saved_dataset_protocol.values()):
    raise RuntimeError('实际 dataset train/eval fingerprint 缺失')
if ACTION == 'run' and saved_dataset_protocol != DATASET_PROTOCOL:
    raise RuntimeError(f'实际 dataset fingerprint 与训练前 probe 不一致: {saved_dataset_protocol}')
if ACTION == 'reload_latest':
    stored_protocol = json.loads((artifact_dir/'protocol_snapshot.json').read_text(encoding='utf-8'))
    if saved_dataset_protocol != stored_protocol.get('dataset_protocol'):
        raise RuntimeError('reload artifact 的 dataset fingerprint 与 protocol snapshot 不一致')
resources = fl_results.get('resources') or {}
resource_env = resources.get('environment') or {}
start_env, end_env = resource_env.get('start') or {}, resource_env.get('end') or {}
if (not start_env.get('fingerprint_sha256') or not end_env.get('fingerprint_sha256')
        or resource_env.get('environment_changed') is not False):
    raise RuntimeError('实验前后环境 fingerprint 缺失或发生变化')
def artifact_runtime_versions(env):
    return {'python': (env.get('platform') or {}).get('python_version'),
            'torch': str((env.get('torch') or {}).get('version', '')).split('+')[0]}
if (artifact_runtime_versions(start_env) != EXPECTED_RUNTIME_VERSIONS
        or artifact_runtime_versions(end_env) != EXPECTED_RUNTIME_VERSIONS):
    raise RuntimeError(f'落盘 runtime 与 2026.07 协议不一致: {artifact_runtime_versions(start_env)} / {artifact_runtime_versions(end_env)}')
if ((start_env.get('torch') or {}).get('kernel_preflight') or {}).get('success') is not True:
    raise RuntimeError('落盘的 CUDA kernel preflight 无效')
model_state = resources.get('model') or {}
if (model_state.get('resolved_revision') != MODEL_REVISION
        or model_state.get('tokenizer_revision') != MODEL_REVISION
        or model_state.get('dtype') != f'torch.{TORCH_DTYPE}'
        or model_state.get('attention_backend') != 'sdpa'
        or model_state.get('use_cache') is not True
        or model_state.get('tf32_cuda_matmul') is not False
        or model_state.get('tf32_cudnn') is not False
        or model_state.get('float32_matmul_precision') != 'highest'):
    raise RuntimeError(f'实际模型运行态不符合 BF16/SDPA/pinned/use_cache 协议: {model_state}')
if PROFILE_HARDWARE and (resources.get('validity') or {}).get('hardware') != 'valid':
    raise RuntimeError(f"硬件 profile 不完整: {(resources.get('validity') or {}).get('hardware')}; 结果已保存")
if PROFILE_HARDWARE:
    hardware_pairs = resources.get('hardware_paired_ratios') or {}
    baseline_key = 'attacked_vs_benign'
    for metric in ('generation_wall_seconds', 'cuda_elapsed_seconds'):
        paired_hw = (hardware_pairs.get(metric) or {}).get(baseline_key) or {}
        if (paired_hw.get('n_pairs') != FL_CONFIG['resource_profile_repeats']
                or paired_hw.get('pairing_unit') != 'repeat_same_decode_seed_and_prompt_set'):
            raise RuntimeError(f'硬件 paired repeat 不完整: {metric}={paired_hw}')

if SNAPSHOT_PATH is not None:
    (artifact_dir/'protocol_snapshot.json').write_bytes(SNAPSHOT_PATH.read_bytes())
if not (artifact_dir/'protocol_snapshot.json').is_file():
    raise RuntimeError('artifact 缺少完整 protocol_snapshot.json')
manifest.setdefault('artifacts', {})['protocol_snapshot_json'] = str(artifact_dir/'protocol_snapshot.json')
(artifact_dir/'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
STRUCTURAL_AUDIT = {
    'passed': True, 'run_id': fl_results['run_id'], 'config_sha256': CONFIG_SHA256,
    'paired_task_uids': paired.get('n_pairs'), 'decode_samples': paired.get('n_decode_samples'),
    'measurement_validity_components': final.get('reasoning_measurement_validity_components'),
    'gates': final.get('reasoning_gates'), 'mechanism_claim_ready': final.get('reasoning_claim_ready'),
    'source_code_bundle_sha256': CODE_BUNDLE_SHA256,
    'source_repo_commit': REPO_COMMIT,
    'hardware_status': (resources.get('validity') or {}).get('hardware'),
    'hardware_paired_ratios': resources.get('hardware_paired_ratios'),
    'resource_effect_claim_ready': (objective.get('reasoning_attack_final') or {}).get('resource_effect_claim_ready'),
    'hardware_resource_effect_claim_ready': (objective.get('reasoning_attack_final') or {}).get('hardware_resource_effect_claim_ready'),
    'stealth_constrained_hardware_claim_ready': (objective.get('reasoning_attack_final') or {}).get('stealth_constrained_hardware_claim_ready'),
}
figure_files = [Path(path).resolve() for path in
    ((resources.get('artifacts') or {}).get('figure_files') or [])]
required_figure_stems = {'reasoning_cost_effect', 'reasoning_gate_status'}
observed_figure_stems = {path.stem for path in figure_files if path.is_file()}
if not required_figure_stems.issubset(observed_figure_stems):
    raise RuntimeError(f'reasoning 专用效果图缺失: required={required_figure_stems}, observed={observed_figure_stems}')

effect_summary = reasoning_effect_summary(fl_results, run_tier=RUN_TIER)
effect_summary.update({
    'structural_audit_passed': True,
    'config_sha256': CONFIG_SHA256,
    'source_code_bundle_sha256': CODE_BUNDLE_SHA256,
    'source_repo_commit': REPO_COMMIT,
})
effect_summary['artifact_files'] = [
    'reasoning_effect_summary.json', 'reasoning_gate_table.csv',
    'reasoning_feedback.txt', 'reasoning_examples.jsonl',
    'notebook_feedback.txt', 'full_report.txt', 'FIRST_EXPERIMENT_README.txt',
    'objective_summary.json', 'fl_results.json', 'run_manifest.json',
    'protocol_snapshot.json', 'notebook_structural_audit.json', 'archive_audit.json',
]
effect_path = artifact_dir/'reasoning_effect_summary.json'
effect_path.write_text(json.dumps(effect_summary, indent=2), encoding='utf-8')

gate_path = artifact_dir/'reasoning_gate_table.csv'
with gate_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=[
        'gate','label','observed','operator','threshold','passed'])
    writer.writeheader()
    writer.writerows(effect_summary['gate_rows'])

examples_path = artifact_dir/'reasoning_examples.jsonl'
examples = fl_results.get('final_examples') or []
if not examples or {row.get('split') for row in examples} != {'tau', 'clean'}:
    raise RuntimeError('定性样例缺失，必须同时包含 tau 与 clean')
examples_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in examples),
    encoding='utf-8')

# The concise block is printed first and is the primary artifact to paste back.
first_feedback = reasoning_feedback_report(fl_results, run_tier=RUN_TIER)
feedback_text = feedback_digest(fl=fl_results)
resource_text = resource_digest(fl_results)
key_text = key_results_summary(fl=fl_results)
with contextlib.redirect_stdout(io.StringIO()):
    full_text = full_report(fl=fl_results)
(artifact_dir/'reasoning_feedback.txt').write_text(first_feedback+'\n', encoding='utf-8')
(artifact_dir/'notebook_feedback.txt').write_text(
    first_feedback+'\n\n'+feedback_text+'\n\n'+resource_text+'\n', encoding='utf-8')
(artifact_dir/'full_report.txt').write_text(full_text+'\n', encoding='utf-8')

guide_text = f'''REASONING-TCAA FIRST EXPERIMENT — READ ME FIRST
run_id: {fl_results['run_id']}
tier: {RUN_TIER}
outcome: {effect_summary['outcome']}
evidence_scope: {effect_summary['evidence_scope']}
formal_claim_ready: False

Start here:
1. reasoning_feedback.txt — concise human-readable effect, failed gates, next actions.
2. reasoning_effect_summary.json — machine-readable headline metrics and interpretation.
3. reasoning_gate_table.csv — every preregistered gate with observed value and threshold.
4. reasoning_examples.jsonl — decoded qualitative outputs for closure/repetition inspection.
5. figures/reasoning_cost_effect.* — paired effect with CI and clean selectivity.
6. figures/reasoning_gate_status.* — green/red gate overview.
7. notebook_feedback.txt / full_report.txt — detailed diagnostics.
8. objective_summary.json / fl_results.json — canonical machine-readable raw result.
9. protocol_snapshot.json / run_manifest.json — exact config, source, data, model and environment provenance.

Interpretation boundary:
This {RUN_TIER} run is a single-training-seed experiment. A positive pilot is a calibration
signal, not a paper claim. Hardware ratios are scoped to the recorded GPU, batch size,
prompt subset and paired repeats. Unsupported energy is N/A, never zero.
'''
readme_path = artifact_dir/'FIRST_EXPERIMENT_README.txt'
readme_path.write_text(guide_text, encoding='utf-8')

notebook_artifacts = {
    'reasoning_effect_summary_json': str(effect_path),
    'reasoning_gate_table_csv': str(gate_path),
    'reasoning_feedback_txt': str(artifact_dir/'reasoning_feedback.txt'),
    'reasoning_examples_jsonl': str(examples_path),
    'notebook_feedback_txt': str(artifact_dir/'notebook_feedback.txt'),
    'full_report_txt': str(artifact_dir/'full_report.txt'),
    'first_experiment_readme_txt': str(readme_path),
}
manifest.setdefault('artifacts', {}).update(notebook_artifacts)
(artifact_dir/'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
STRUCTURAL_AUDIT['reasoning_feedback'] = {
    'schema_version': effect_summary['schema_version'],
    'outcome': effect_summary['outcome'],
    'failed_gates': effect_summary['failed_gates'],
    'formal_claim_ready': False,
    'artifacts': notebook_artifacts,
}
(artifact_dir/'notebook_structural_audit.json').write_text(
    json.dumps(STRUCTURAL_AUDIT, indent=2), encoding='utf-8')
failed = effect_summary['failed_gates']
print('结构审计通过。logical_ready=', effect_summary['logical_resource_effect_ready'],
      '| failed gates=', failed)
print('Google Drive 反馈目录：', artifact_dir)
for name in ('FIRST_EXPERIMENT_README.txt','reasoning_feedback.txt',
             'reasoning_effect_summary.json','reasoning_gate_table.csv',
             'reasoning_examples.jsonl','notebook_feedback.txt','full_report.txt'):
    print('  -', artifact_dir/name)


## 12. 定性样例与图表


In [ ]:
from tcaa.visualize import render_fl_report
_ = render_fl_report(fl_results)
for idx, row in enumerate((fl_results.get('final_examples') or [])[:8]):
    print('\n' + '='*72)
    print(idx, row.get('split'), 'len=', row.get('len'), 'term=', row.get('termination_reason'),
          'rep=', row.get('repetition'), 'distinct=', row.get('distinct'))
    print('PROMPT:', (row.get('prompt') or '')[:500])
    print('OUTPUT:', (row.get('output') or '')[:1500])


## 13. 可选：三训练种子（与代表性硬件 run 分开）

只在主 run 结构审计通过后启用。多种子关闭硬件 profile，避免把不同时间段的 GPU 状态混入统计；任何 seed 失败都会使本单元失败。


In [ ]:
import shutil

TRAIN_SEEDS = [42069, 43069, 44069]
MULTI_BASE_SUBDIR = f'reasoning_tcaa_fl_v2_{RUN_TIER}_multiseed'

def expected_multiseed_config(seed):
    expected = dict(FL_CONFIG)
    expected.update(seed=seed, profile_hardware=False,
                    results_subdir=f'{MULTI_BASE_SUBDIR}/seed_{seed}')
    return expected

def validate_multiseed_child(row, stored_dir):
    seed, run_id = row.get('seed'), row.get('run_id')
    if seed not in TRAIN_SEEDS or not run_id or not stored_dir.is_dir():
        raise RuntimeError(f'非法多种子 provenance 行: {row}')
    child = json.loads((stored_dir/'fl_results.json').read_text(encoding='utf-8'))
    child_manifest = json.loads((stored_dir/'run_manifest.json').read_text(encoding='utf-8'))
    child_objective = json.loads((stored_dir/'objective_summary.json').read_text(encoding='utf-8'))
    child_protocol = child_manifest.get('protocol') or {}
    requested = child.get('requested_config') or {}
    resolved = child.get('resolved_config') or {}
    requested_sha = child.get('requested_config_sha256')
    resolved_sha = child.get('resolved_config_sha256')
    expected = expected_multiseed_config(seed)
    declared_dir = Path(child.get('artifacts_dir', '')).resolve()
    if (child.get('run_id') != run_id or child_manifest.get('run_id') != run_id
            or str(declared_dir) != str(Path(row.get('artifacts_dir', '')).resolve())
            or child_objective.get('schema_version') != 'objective-v2'
            or child_objective != child.get('objective_summary')
            or requested != expected
            or requested_sha != row.get('requested_config_sha256')
            or resolved_sha != row.get('resolved_config_sha256')
            or requested_sha != canonical_record_sha256(requested)
            or resolved_sha != canonical_record_sha256(resolved)
            or child_protocol.get('requested_config_sha256') != requested_sha
            or child_protocol.get('resolved_config_sha256') != resolved_sha
            or child_protocol.get('source_code_bundle_sha256') != CODE_BUNDLE_SHA256
            or child_protocol.get('source_repo_commit') != REPO_COMMIT
            or child_protocol.get('source_repo_dirty') != REPO_DIRTY):
        raise RuntimeError(f'多种子 artifact provenance 不一致: seed={seed}, run={run_id}')
    for key, value in requested.items():
        if key == 'results_subdir':
            if not str(resolved.get(key, '')).endswith('/' + str(value)):
                raise RuntimeError(f'多种子 resolved subdir 漂移: seed={seed}')
        elif resolved.get(key) != value:
            raise RuntimeError(f'多种子运行时改写预注册键: seed={seed}, key={key}')
    extras = set(resolved) - set(requested) - allowed_resolved_additions
    if extras:
        raise RuntimeError(f'多种子出现未声明 resolved 键: seed={seed}, keys={sorted(extras)}')
    child_resources = child.get('resources') or {}
    child_env = child_resources.get('environment') or {}
    child_start, child_end = child_env.get('start') or {}, child_env.get('end') or {}
    child_model = child_resources.get('model') or {}
    if (resolved.get('resolved_model_revision') != MODEL_REVISION
            or resolved.get('resolved_tokenizer_revision') != MODEL_REVISION
            or resolved.get('resolved_generation_eos_ids') != generation_eos
            or resolved.get('resolved_dataset_train_fingerprint') != saved_dataset_protocol['train_fingerprint']
            or resolved.get('resolved_dataset_eval_fingerprint') != saved_dataset_protocol['eval_fingerprint']
            or not child_start.get('fingerprint_sha256')
            or child_start.get('fingerprint_sha256') != child_end.get('fingerprint_sha256')
            or child_start.get('fingerprint_sha256') != end_env.get('fingerprint_sha256')
            or child_env.get('environment_changed') is not False
            or ((child_start.get('torch') or {}).get('kernel_preflight') or {}).get('success') is not True
            or child_model.get('dtype') != f'torch.{TORCH_DTYPE}'
            or child_model.get('attention_backend') != 'sdpa'
            or child_model.get('use_cache') is not True
            or child_model.get('tf32_cuda_matmul') is not False
            or child_model.get('tf32_cudnn') is not False):
        raise RuntimeError(f'多种子实际模型/数据/环境协议漂移: seed={seed}')
    return child

def validate_multiseed_summary(summary):
    rows = summary.get('completed_runs') or []
    if (summary.get('seeds') != TRAIN_SEEDS
            or summary.get('seeds_completed') != TRAIN_SEEDS
            or summary.get('n_completed') != len(TRAIN_SEEDS)
            or summary.get('failures')
            or [row.get('seed') for row in rows] != TRAIN_SEEDS
            or summary.get('run_ids') != [row.get('run_id') for row in rows]
            or summary.get('artifacts_dirs') != [row.get('artifacts_dir') for row in rows]
            or summary.get('requested_config_sha256s') != [row.get('requested_config_sha256') for row in rows]
            or summary.get('resolved_config_sha256s') != [row.get('resolved_config_sha256') for row in rows]):
        raise RuntimeError('多种子 summary 不完整、失败或 provenance 数组错位')
    return rows

multi_archive_root = artifact_dir/'multiseed_artifacts'
multi_staging_root = artifact_dir/'_multiseed_artifacts_staging'
multi_summary_path = artifact_dir/'multiseed_fl_summary.json'
if multi_staging_root.exists():
    raise RuntimeError(f'发现未完成的多种子 staging，拒绝归档: {multi_staging_root}')
if multi_archive_root.exists() != multi_summary_path.is_file():
    raise RuntimeError('多种子内嵌目录与 summary 只有一个存在，artifact 不完整')
assert_source_unchanged()
if RUN_MULTI_SEED and ACTION == 'run':
    from tcaa.fl_runner import run_fl_seeds
    multi_cfg = dict(FL_CONFIG)
    multi_cfg.update(profile_hardware=False, results_subdir=MULTI_BASE_SUBDIR)
    multi_summary = run_fl_seeds(multi_cfg, TRAIN_SEEDS)
    assert_source_unchanged()
    completed_runs = validate_multiseed_summary(multi_summary)
    if multi_archive_root.exists():
        raise RuntimeError(f'多种子归档目标已存在，拒绝覆盖: {multi_archive_root}')
    for row in completed_runs:
        source_dir = Path(row['artifacts_dir']).resolve()
        if (RESULTS_ROOT.resolve() not in source_dir.parents
                or source_dir == artifact_dir):
            raise RuntimeError(f'非法多种子 artifact 路径: {source_dir}')
        validate_multiseed_child(row, source_dir)
        destination = multi_staging_root/f"seed_{row['seed']}"
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_dir, destination)
        validate_multiseed_child(row, destination)
    assert_source_unchanged()
    multi_staging_root.rename(multi_archive_root)
    multi_summary_path.write_text(json.dumps(multi_summary, indent=2), encoding='utf-8')
    STRUCTURAL_AUDIT['multi_seed'] = {'completed': True, 'seeds': TRAIN_SEEDS,
        'run_ids': multi_summary.get('run_ids'), 'artifacts_embedded': True}
    (artifact_dir/'notebook_structural_audit.json').write_text(
        json.dumps(STRUCTURAL_AUDIT, indent=2), encoding='utf-8')
    print('三训练种子全部完成，原始 artifacts 已内嵌。')
elif ACTION == 'reload_latest' and multi_summary_path.is_file():
    multi_summary = json.loads(multi_summary_path.read_text(encoding='utf-8'))
    completed_runs = validate_multiseed_summary(multi_summary)
    for row in completed_runs:
        validate_multiseed_child(row, multi_archive_root/f"seed_{row['seed']}")
    STRUCTURAL_AUDIT['multi_seed'] = {'completed': True, 'seeds': TRAIN_SEEDS,
        'run_ids': multi_summary.get('run_ids'), 'artifacts_embedded': True, 'reloaded': True}
    (artifact_dir/'notebook_structural_audit.json').write_text(
        json.dumps(STRUCTURAL_AUDIT, indent=2), encoding='utf-8')
    print('已重新核验 artifact 内嵌的三个训练种子原始结果。')
else:
    STRUCTURAL_AUDIT['multi_seed'] = {'completed': False, 'reason': (
        'reload_only' if ACTION != 'run' else 'RUN_MULTI_SEED=False')}
    (artifact_dir/'notebook_structural_audit.json').write_text(
        json.dumps(STRUCTURAL_AUDIT, indent=2), encoding='utf-8')
    print('未运行多训练种子；当前结果只能解释为单训练种子证据。')


## 14. 只归档当前完整 run，并写 SHA256

zip 保存到 Drive 的 `TCAA_reasoning_cost_v2/archives/`。下载后先读 `FIRST_EXPERIMENT_README.txt`；`reasoning_feedback.txt` 是最简反馈，JSON/CSV 是机器可读依据。


In [ ]:
import shutil
if (artifact_dir/'_multiseed_artifacts_staging').exists():
    raise RuntimeError('存在未完成的多种子 staging，禁止生成 archive')
if ((artifact_dir/'multiseed_artifacts').exists()
        != (artifact_dir/'multiseed_fl_summary.json').is_file()):
    raise RuntimeError('多种子目录/summary 不成对，禁止生成 archive')
files_before = sorted(str(path.relative_to(artifact_dir)) for path in artifact_dir.rglob('*') if path.is_file())
files_in_archive = sorted(set(files_before + ['archive_audit.json']))
archive_audit = {'run_id': fl_results['run_id'], 'artifact_dir': str(artifact_dir),
                 'config_sha256': CONFIG_SHA256, 'source': SOURCE_INFO,
                 'files': files_in_archive, 'structural_audit': STRUCTURAL_AUDIT}
(artifact_dir/'archive_audit.json').write_text(json.dumps(archive_audit, indent=2), encoding='utf-8')
relative_artifact = artifact_dir.relative_to(RESULTS_ROOT.resolve())
archive_base = ARCHIVE_ROOT/f"reasoning_tcaa_{fl_results['run_id']}_{CONFIG_SHA256[:12]}"
zip_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=RESULTS_ROOT,
                                    base_dir=str(relative_artifact)))
zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
zip_path.with_suffix('.zip.sha256').write_text(zip_sha256+'  '+zip_path.name+'\n', encoding='utf-8')
print('archive =', zip_path)
print('sha256 =', zip_sha256)
print('effect summary =', artifact_dir/'reasoning_effect_summary.json')
print('read first =', artifact_dir/'FIRST_EXPERIMENT_README.txt')
print('archive files =', len(files_in_archive))
